# OpenPyTEA — Tutorial 1: Creating Equipment

The `Equipment` class is the building block of every OpenPyTEA analysis. It represents a single piece of process equipment and handles:

- **Purchased cost estimation** — from built-in cost correlations or a user-supplied value
- **Inflation adjustment** — using the Chemical Engineering Plant Cost Index (CEPCI)
- **Installed (direct) cost calculation** — by applying material and installation cost factors

This tutorial uses a set of **representative process equipment** as the running example and walks through the main ways to create and inspect `Equipment` objects.

---

In [ ]:
from openpytea import Equipment

## 1. Browse the Reference Databases

OpenPyTEA ships with two reference databases, both as plain CSV files you can open in any spreadsheet application.

### Cost correlations — `cost_correlation.csv`

Contains built-in cost correlations drawn from standard chemical engineering references (Turton et al., Towler & Sinnott, Parkinson, and others). The key columns are:

| Column | Description |
|---|---|
| `category` | Equipment category (e.g. `Compressors, fans, & blowers`) |
| `type` | Sub-type within the category (e.g. `Compressor, centrifugal`) |
| `units` | Sizing parameter units (e.g. `kW`, `m²`) |
| `s_lower` / `s_upper` | Valid sizing range — stay within these bounds |

<br>

> **Extensible.** Append rows to `cost_correlation.csv` and OpenPyTEA picks them up automatically. If you find a useful correlation in the literature, we strongly encourage you to [submit it to the project](https://github.com/openpytea/openpytea/issues) so the whole community benefits.

### CEPCI values — `cepci_values.csv`

Contains the Chemical Engineering Plant Cost Index (CEPCI) values used to inflate or deflate costs between years. The key columns are:

| Column | Description |
|---|---|
| `year` | Calendar year |
| `cepci` | Published CEPCI index value for that year |

<br>

> **Extensible.** If you have access to more recent CEPCI values, or want to use an alternative cost index, simply add or replace rows in `cepci_values.csv`. OpenPyTEA will use whatever values are present in the file.

## 2. Create Equipment Using a Database Correlation

Supply a **sizing parameter** (e.g., power in kW or area in m²) and OpenPyTEA looks up the purchased cost from its correlation database.

| Argument | Type | Description |
|---|---|---|
| `name` | str | Equipment tag or label (e.g., `"COMP-1"`) |
| `param` | float | Sizing parameter — value and units depend on the equipment type |
| `process_type` | str | `"Fluids"`, `"Solids"`, `"Mixed"`, or `"Electrical"` — sets default installation factors |
| `category` | str | Equipment category from the database |
| `type` | str | Equipment sub-type within the category (optional but recommended for accuracy) |
| `material` | str | Material of construction (default: `"Carbon steel"`) |
| `cost_func` | str | Exact correlation key — use when multiple entries share the same `category` and `type` |

In [ ]:
# Feed gas compressor — 200 kW centrifugal
compressor = Equipment(
    name="COMP-1",
    param=200,                                    # driver power, kW
    process_type="Fluids",
    category="Compressors, fans, & blowers",
    type="Compressor, centrifugal",
    material="Carbon steel",
)

print(compressor)

### Selecting a Specific Correlation with `cost_func`

Some equipment types have **more than one correlation** in the database — for example, different sources in the literature may provide separate fits for the same type. By default, OpenPyTEA uses the **first matching entry** it finds in `cost_correlation.csv`.

To target a specific correlation explicitly, pass its key via the `cost_func` argument. The key is listed in the `cost_func` column of `cost_correlation.csv`.

The example below shows the same filter created twice with two different correlation keys:

In [ ]:
# Default behaviour — takes the first matching entry in the database
filter_default = Equipment(
    name="FILT-default",
    param=10, # m3                                    # check units in cost_correlation.csv
    process_type="Fluids",
    category="Filters",
    type="Plate & frame",
    material="Carbon steel",
)

# Explicit correlation — targets a specific entry by its cost_func key
filter_explicit = Equipment(
    name="FILT-explicit",
    param=2000, # m2                                   # check units in cost_correlation.csv
    process_type="Fluids",
    category="Filters",
    type="Plate & frame",
    material="Carbon steel",
    cost_func="plate_frame_filter_turton_2001",        # replace with key from cost_correlation.csv
)

print("FILT-default (first match in database)")
print(f"  Purchased cost : ${filter_default.purchased_cost:>12,.0f}")
print("\nFILT-explicit (targeted correlation)")
print(f"  Purchased cost : ${filter_explicit.purchased_cost:>12,.0f}")

## 3. Specify a Material of Construction

Different materials carry a cost premium over carbon steel, captured by the **material factor** (`fm`). Common choices include:

- `"Carbon steel"` (fm = 1.0, baseline)
- `"304 stainless steel"` (fm ≈ 1.3)
- `"316 stainless steel"` (fm ≈ 1.5)
- `"Hastelloy C"` (fm ≈ 2.6)

The exact factors depend on the equipment type — OpenPyTEA applies the standard values from Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*.

In [ ]:
# heat exchanger — U-tube shell-and-tube in carbon steel
heat_exchanger_1 = Equipment(
    name="HX-1",
    param=35,                                     # heat transfer area, m²
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="Carbon steel",
)

print(heat_exchanger_1)

In [62]:
# heat exchanger — U-tube shell-and-tube in 304 stainless steel
heat_exchanger_2 = Equipment(
    name="HX-2",
    param=35,                                     # heat transfer area, m²
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="304 stainless steel",
)

print(heat_exchanger_2)

Name=HX-2, Category=Heat Exchangers, Sub-type=U-tube shell & tube, Material=304 stainless steel, Process Type=Fluids, Parameter=35, Number of units=1, Purchased Cost=46257.599896543725, Direct Cost=173003.42361307354)


## 4. Create Equipment with a Known Purchase Cost

When you have a **vendor quote** or a value from literature, bypass the database entirely. Provide `purchased_cost` (in the currency of `cost_year`) and OpenPyTEA inflates it to `target_year` using the CEPCI.

> **Tip:** Set `param=0` when using a direct purchase cost — the sizing parameter is not used for cost lookup.

In [63]:
# Reactor — vendor-quoted cost from 2019
reactor = Equipment(
    name="REACT-1",
    param=0,                                      # unused when purchased_cost is given
    process_type="Fluids",
    category="Reactors",
    purchased_cost=2_500_000,                     # USD (2019 value)
    cost_year=2019,                               # year of the quoted cost
    target_year=2024,                             # inflate to this year
)

print(reactor)

Name=REACT-1, Category=Reactors, Sub-type=None, Material=Carbon steel, Process Type=Fluids, Parameter=None, Number of units=1, Purchased Cost=3292181.069958848, Direct Cost=10534979.423868313)


## 5. Access Equipment Attributes

After creating an `Equipment` object you can read key computed values as Python attributes.

In [64]:
print("COMP-1 (from database correlation)")
print(f"  Purchased cost (2024 USD) : ${compressor.purchased_cost:>12,.0f}")
print(f"  Direct cost    (2024 USD) : ${compressor.direct_cost:>12,.0f}")
print(f"  Material factor (fm)      : {compressor.material_factor:>12.2f}")
print(f"  Number of units           : {compressor.num_units:>12d}")

print("\nREACT-1 (direct cost, inflated from 2019)")
print(f"  Purchased cost (2024 USD) : ${reactor.purchased_cost:>12,.0f}")
print(f"  Direct cost    (2024 USD) : ${reactor.direct_cost:>12,.0f}")

COMP-1 (from database correlation)
  Purchased cost (2024 USD) : $   1,540,232
  Direct cost    (2024 USD) : $   4,928,743
  Material factor (fm)      :         1.00
  Number of units           :            1

REACT-1 (direct cost, inflated from 2019)
  Purchased cost (2024 USD) : $   3,292,181
  Direct cost    (2024 USD) : $  10,534,979


## 6. Customise Installation Cost Factors

The `process_type` selects a default set of installation factors (erection, piping, instrumentation, civil, structural, lagging) based on Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*.

Override individual factors when you have project-specific data:

In [65]:
# Same compressor with a higher piping complexity
compressor_high_piping = Equipment(
    name="COMP-1 (high piping)",
    param=200,
    process_type="Fluids",
    category="Compressors, fans, & blowers",
    type="Compressor, centrifugal",
    material="Carbon steel",
    piping_factor=1.5,         # override default piping factor
    erection_factor=0.5,       # override default erection factor
)

print(f"Standard direct cost : ${compressor.direct_cost:>12,.0f}")
print(f"High-piping cost     : ${compressor_high_piping.direct_cost:>12,.0f}")

Standard direct cost : $   4,928,743
High-piping cost     : $   6,314,951


## Summary

Two approaches to creating an `Equipment` object:

| Method | When to use |
|---|---|
| Database lookup (`param`) | Equipment size is known; the type exists in the built-in database |
| Direct cost (`purchased_cost` + `cost_year`) | A vendor quote or literature value is available |

**Next:** Tutorial 2 — Creating a Plant shows how to collect these equipment objects into a `Plant` and compute full techno-economic metrics.